In [30]:
import requests
import lxml.html
from pypdf import PdfReader
import pandas as pd
from google import genai
from pathlib import Path
from google.genai import types
import os
import us
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

In [ ]:
# from ingest_sc_cases import case_df
# from ingest_sc_judges import judge_pd

In [31]:
case_df = pd.read_csv("../data/cases_scdb.csv")
judge_pd = pd.read_csv("../data/judges_slri.csv")

In [32]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,Unnamed: 0,docket_no,title,state,date,type,pending,opinion_link
12,12,2026-00384,Williams v. Board of Elections of the State of...,New York,"March 19, 2026",Voting Rights and Elections,False,NaN
37,37,20220896,State v. Andrus,Utah,"August 7, 2025",Criminal Law,False,NaN
49,49,S-1-SC-40715,Franklin v. Martinez,New Mexico,"January 26, 2026",Criminal Law,False,NaN
101,101,22 OC 00022 1 B,Persaud-Zamora v. Cegavske,Nevada,"April 26, 2022",Voting Rights and Elections,False,NaN
105,105,EF2025-2536,Texas v. Bruck,Texas,"October 31, 2025",Reproductive Rights,False,NaN
107,107,2025AP002121,Voces de la Frontera v. Gerber,Wisconsin,"September 18, 2025",Government Structure,False,NaN
124,124,PR-123179,McVay v. Cockroft,Oklahoma,"July 28, 2025",Voting Rights and Elections,False,NaN
156,156,15-25-00093-CV,State v. City of San Antonio,Texas,"June 19, 2025",Reproductive Rights,False,NaN
210,210,CV01-23-14744,Adkins v. State,Idaho,"April 11, 2025",Reproductive Rights,False,NaN
232,232,SJC-13666,Gotay v. Creen,Massachusetts,"March 21, 2025",Civil Due Process,False,NaN


In [34]:
case_df["type"].unique()

<StringArray>
[                         'Civil Rights',
                  'Government Structure',
             'Economic and Labor Rights',
           'Voting Rights and Elections',
                          'Criminal Law',
                           'Environment',
 'Judicial Selection and Administration',
                             'Education',
                   'Speech and Religion',
                     'Civil Due Process',
                   'Reproductive Rights',
                   'Torts and Liability',
               'Judicial Interpretation',
                         'Election 2024',
                                     nan]
Length: 15, dtype: str

In [ ]:
class IndividualOpinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    ruling: str = Field(
        description='"concur" or "dissent" or "other" based on how this judge ruled,'
    )
    description: str = Field(
        description="An extremely brief description of the judge's own opinion on the case."
    )


class Case(BaseModel):
    # title: str = Field(description="The title of the case.")
    issue_debate: str = Field(
        description='A phrase starting with "Whether" that summarizes '
        "the main issue being debated in the case"
    )
    plaintiff_argument: str = Field(
        description="Briefly, the plaintiff's stance on the debate issue"
    )
    plaintiff_pro_con: str = Field(
        description="Whether the plaintiff's argument is generally pro or con for the debate issue"
    )
    defendant_argument: str = Field(
        description="Briefly, the defendant's stance on the debate issue"
    )
    decision_outcome: str = Field(
        description="The court's final decision for the case whether "
        "they ruled with the plaintiff or not."
    )
    judge_opinions: List[IndividualOpinion]

In [11]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text)

In [12]:
def analyze_state_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)
    num_cases = len(case_df)
    model_id = client_info["model_id"]
    client = client_info["client"]

    file_dic = {}
    for i in range(num_cases):
        print(f"Querying case {i}")
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        pdf_link = case_df.iloc[i]["opinion_link"]

        case_ref = "_".join([docket_no, state, date])

        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp.model_dump()

    return file_dic

In [13]:
def apply_model(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()

    load_dotenv(find_dotenv())

    gemini_key = os.getenv("GEMINI_API_KEY")
    client_info = {"model_id": "gemini-2.5-flash-lite", "client": genai.Client(api_key=gemini_key)}

    case_dic = analyze_state_cases(case_df, judge_df, prompt_start, client_info)

    return case_dic

In [ ]:
def state_opinion_table(case_dic: dict):
    opinion_table = {"case_id": [], "name": [], "opinion": [], "ruling": []}

    for key in case_dic.keys():
        opinions = case_dic[key]["response"]
        num_opinions = len(opinions["judge_opinions"])

        for i in range(num_opinions):
            opinion_table["case_id"].append(key)

            judge_name = opinions["judge_opinions"][i]["judge_name"]
            opinion_table["name"].append(judge_name)

            description = opinions["judge_opinions"][i]["description"]
            opinion_table["opinion"].append(description)

            ruling = opinions["judge_opinions"][i]["ruling"]
            opinion_table["ruling"].append(ruling)

    return pd.DataFrame(opinion_table)

In [ ]:
def state_case_table(case_df: pd.DataFrame, case_dic: dict):
    case_table = {
        "case_id": [],
        "docket_no": [],
        "title": [],
        "state": [],
        "date": [],
        "type": [],
        "description": [],
        "plaintiff_argument": [],
        "plaintiff_pro_con": [],
        "defendant_argument": [],
        "decision_outcome": [],
        "decision_status": [],
    }
    num_cases = len(case_df)

    for i in range(num_cases):
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        case_table["docket_no"].append(docket_no)
        state = case_df.iloc[i]["state"]
        case_table["state"].append(state)
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_table["date"].append(date)

        title = case_df.iloc[i]["title"]
        case_table["title"].append(title)
        type = case_df.iloc[i]["type"]
        case_table["type"].append(type)

        case_id = "_".join([docket_no, state, date])
        case_table["case_id"].append(case_id)

        response = case_dic[case_id]["response"]
        case_table["description"].append(response["issue_debate"])
        case_table["plaintiff_argument"].append(response["plaintiff_argument"])
        case_table["plaintiff_pro_con"].append(response["plaintiff_pro_con"])
        case_table["defendant_argument"].append(response["defendant_argument"])
        case_table["decision_outcome"].append(response["decision_outcome"])

        status = ~case_df.iloc[i]["pending"]  # Negate, "not pending" means "decided"
        case_table["decision_status"].append(status)

    return pd.DataFrame(case_table)

In [16]:
def produce_tables(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    states = case_df["state"].sort_values().unique()

    opinions = []
    cases = []

    for state in states:
        state_cases = case_df[case_df["state"] == state]

        state_abbr = str(us.states.lookup(state).abbr)
        state_judges = judge_df[judge_df["state"] == state_abbr]

        case_dic = apply_model(state_cases, state_judges, prompt_path)

        opinions.append(state_opinion_table(case_dic))
        cases.append(state_case_table(state_cases, case_dic))

    rd = {"opinion_table": pd.concat(opinions), "case_table": pd.concat(cases)}

    return rd

In [24]:
mon_cases = case_df[case_df["state"] == "Montana"].tail(5)
mon_judges = judge_pd[judge_pd["state"] == "MT"]

mon_opinions = produce_tables(mon_cases, mon_judges, "prompt.txt")

Querying case 0
Querying case 1


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 38.320666502s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-fla

In [ ]:
display(mon_opinions["opinion_table"])
display(mon_opinions["case_table"])

,case_id,name,opinion
0,25-0139_Montana_2026/04/14,Laurie McKinnon,Affirmed the District Court's order preliminar...
1,25-0139_Montana_2026/04/14,James Jeremiah Shea,Concurred with the majority opinion affirming ...
2,25-0139_Montana_2026/04/14,Ingrid Gustafson,Concurred with the majority opinion affirming ...
3,25-0139_Montana_2026/04/14,Katherine Biidegaray,Concurred with the majority opinion affirming ...
4,25-0139_Montana_2026/04/14,Beth Baker,"Concurred narrowly, agreeing to uphold the pre..."
5,25-0139_Montana_2026/04/14,James A. Rice,"Dissented, arguing that the decision forces th..."
6,25-0139_Montana_2026/04/14,Cory Swanson,"Joined Justice Rice's dissent, expressing conc..."
7,OP-25-0858_Montana_2026/02/27,Katherine M. Bidegaray,"Delivered the Opinion and Order of the Court, ..."
8,OP-25-0858_Montana_2026/02/27,James Jeremiah Shea,Concurred with the Court's decision.
9,OP-25-0858_Montana_2026/02/27,Laurie McKinnon,Concurred with the Court's decision.


,case_id,docket_no,title,state,date,type,dispute_desc,decision_outcome,decision_status
0,25-0139_Montana_2026/04/14,25-0139,Kalarchik v. State,Montana,2026/04/14,Civil Rights,The main issue is whether State Policies limit...,The District Court's Order is affirmed.,False
1,OP-25-0858_Montana_2026/02/27,OP-25-0858,Kendrick v. Knudsen,Montana,2026/02/27,Voting Rights and Elections,The issue is whether Ballot Issue 8 (BI-8) vio...,"The petition is granted, and the Attorney Gene...",False
2,PR-23-0496_Montana_2025/12/31,PR-23-0496,In the Matter of Austin Miles Knudsen,Montana,2025/12/31,Government Structure,Whether Attorney General Knudsen violated Mont...,DISMISSED,False
3,DA-24-0039_Montana_2026/03/17,DA-24-0039,Montanans Against Irresponsible Densification ...,Montana,2026/03/17,Civil Rights,The main issue debated is whether the Montana ...,The Court reversed the District Court's declar...,False
4,DA-24-0250_Montana_2025/08/05,DA-24-0250,State v. Alford,Montana,2025/08/05,Criminal Law,The main issue is whether the mandatory minimu...,The District Court did not err when it sentenc...,False
